# Spotify Intelligence System
**End-to-End Data Science & Machine Learning Project**

---
**Phases Covered:**
1. Data Cleaning & Preprocessing
2. Exploratory Data Analysis (EDA)
3. Hit Song Prediction (ML Classification)
4. Music Recommendation System
5. Song Clustering
6. Streamlit App (see `app/streamlit_app.py`)

**Dataset:** [Spotify Tracks Dataset – Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)

##  Install Required Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm shap lime plotly streamlit -q

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, classification_report)


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore")

import os 
import shap
import joblib

print('All libraries imported successfully!')

#### Load the dataset

In [ ]:
df=pd.read_csv("data/dataset.csv")
df.shape
df.head()

#### Drop Null and duplicate values

In [ ]:
df.isnull().sum()
print("duplicated values:",df.duplicated().sum())
df=df.dropna()
df=df.drop_duplicates()

In [ ]:
corr_matrix= df.corr(numeric_only=True)
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap="YlGnBu")
plt.show()

## Phase-1 Feature engineering and encoding

In [ ]:
#  Popularity normalization 
df['popularity'] = df['popularity'] / 100

#  Duration in minutes 
df['duration_mins'] = df["duration_ms"] / 60000

# Encoded explicit column (true=1 | false=0)
df['explicit'] = df['explicit'].astype(int)

# Encoded track_genre 
le=LabelEncoder()
df["encoded_genre"] = le.fit_transform(df['track_genre'])

# Mood score: average of valence and energy
df["mood_score"] = (df["valence"] + df["energy"]) / 2

# Hit label: 1 if popularity>=0.65 else 0
df["hit"] = (df["popularity"] >= 0.65).astype(int)

print("Features engineered and encoded")


## Scaling the features


In [ ]:
#Scale the audio features
features = ['danceability','energy','loudness','speechiness','acousticness','instrumentalness','liveness','valence','tempo','duration_mins']
scaler=MinMaxScaler()

# Make a copy of df 
df_scaled=df.copy()
df_scaled[features]=scaler.fit_transform(df[features])

### Save the clean data

In [ ]:
os.makedirs('data', exist_ok=True)
df.to_csv('data/cleaned_spotify', index=False)
print("new csv created")

## Phase-2 Exploratory Data Analysis

##### Popularity distribution

In [ ]:
# popularity distribution
plt.figure(figsize=(8,6))
sns.histplot(df['popularity'], bins=20, kde=True)
plt.title('Popularity Distribution')
plt.xlabel('Popularity')
plt.ylabel('Frequency')
plt.show()


##### Top 20 genres by average popularity


In [ ]:

top_genres=df.groupby('track_genre')['popularity'].mean().sort_values(ascending=False).head(20)
fig = px.bar(x=top_genres.index, y=top_genres.values,
    title='Top 20 Genres by Average Popularity',
    labels={'x': 'Genre', 'y': 'Average Popularity'},
    color=top_genres.values, 
    color_continuous_scale='Viridis')
fig.show()
plt.show()

#### Distribution of audio features

In [ ]:
df[features].hist(bins=20, figsize=(15,10))
plt.tight_layout()
plt.show()

#### Correlation heatmap 

In [ ]:

corr_features = features + ["popularity"]
corr=df[corr_features].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlGnBu')
plt.tight_layout()
plt.show()

---
## 1. HIT PREDICTION

In [ ]:

X= df[features]
y= df['hit']

print("Hit songs:", y.sum())
print("Non-hit songs:", len(y) - y.sum())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data split into training and testing sets")

print("test set shape:", X_test.shape)
print("training set shape:", X_train.shape)
print('Class ratio:', round(y.mean() * 100, 2), '%')


In [ ]:
#Resampling to handle class imbalance
from imblearn.over_sampling import SMOTE
 
smote = SMOTE(sampling_strategy=0.7, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)


print("After resampling - Hit songs:", y_resampled.sum())
print("After resampling - Non-hit songs:", len(y_resampled) - y_resampled.sum())
print('Class ratio:', round(y_resampled.mean() * 100, 2), '%')

### 1.1 Train multiple models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),   
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state= 42),    
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42)
}
results = []
for name, model in models.items():
    model.fit(X_resampled, y_resampled)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "ROC AUC": round(roc_auc_score(y_test, y_prob), 4)
    })
results_df = pd.DataFrame(results).sort_values(by="ROC AUC", ascending=False)
results_df

    

### 1.2 Classification Report (Random Forest)

In [ ]:

best_model=models['Random Forest']
y_pred=best_model.predict(X_test)  
print("Classification Report for Random Forest:")
print(classification_report(y_test, y_pred, target_names=['Non-Hit', 'Hit']))   


### 1.3 Confusion Matrix

In [ ]:

print("Confusion Matrix for Random Forest:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=['Actual Non-Hit', 'Actual Hit'], columns=['Predicted Non-Hit', 'Predicted Hit'])
cm_df

### 1.4 Feature importance

In [ ]:
importances = pd.DataFrame(best_model.feature_importances_, index=features, columns=['Importance'])
importances

##### TRY custom threshold to improve recall for "hits"

In [ ]:

y_prob = best_model.predict_proba(X_test)[:,1]
y_pred_custom=(y_prob >= 0.3).astype(int)
print("Classification Report for Custom Threshold (0.3) Random Forest:")
print(classification_report(y_test, y_pred_custom, target_names=['Non-Hit', 'Hit']))

#####  TRY Balanced Random Forest to improve recall for "hits"

In [ ]:

rf_balanced=RandomForestClassifier(n_estimators=100, random_state=42,class_weight='balanced')
rf_balanced.fit(X_train, y_train) 
y_pred_balanced=rf_balanced.predict(X_test)
print("Classification Report for Balanced Random Forest:")
print(classification_report(y_test, y_pred_balanced, target_names=['Non-Hit', 'Hit']))

### 1.5 SHAP

In [ ]:
explainer = shap.TreeExplainer(best_model)
X_sample = X_test.sample(500, random_state=42)
shap_values = explainer.shap_values(X_sample)
shap.summary_plot(shap_values, X_sample, feature_names=features)


### 1.6 Save the model

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/hit_predictor.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(X.columns.tolist(), 'models/feature_columns.pkl')
print("dumped the pkl files")

# 2. MUSIC RECOMMENDATION SYSTEM
This is the standard cosine similarity approach — comparing against ALL songs.

In [ ]:
## Select recommendation features 
rec_features = ['danceability','energy','loudness','speechiness','acousticness','instrumentalness','valence','tempo']

# Create recommendation dataframe
df_rec=df[rec_features + ['track_name','artists','track_genre','popularity']].reset_index(drop=True)
df_rec.head()

rec_scaler=MinMaxScaler()
X_recommended = rec_scaler.fit_transform(df_rec[rec_features])
X_recommended.shape

### 2.1 Recommend Top similar songs

In [44]:
def normal_recommendation(song_name, top_n):

    # Find matching song
    matches = df_rec[df_rec['track_name'].str.lower() == song_name.lower()]

    if matches.empty:
        print(f'Song "{song_name}" not found.')
        return None

    # Get song index
    idx = matches.index[0]

    # Get feature vector
    song_vector = X_recommended[idx].reshape(1, -1)

    # Calculate cosine similarity
    sim_scores = cosine_similarity(song_vector, X_recommended)[0]

    # Create temporary dataframe with similarity scores
    temp_df = df_rec.copy()

    temp_df['similarity_score'] = sim_scores

    # Remove the input song itself
    temp_df = temp_df[temp_df['similarity_score'] != 1]

    # Keep only songs with popularity >= 0.75
    temp_df = temp_df[temp_df['popularity'] >= 0.65]

    # Sort by similarity score
    temp_df = temp_df.sort_values(by='similarity_score', ascending=False) 

    # Remove repeated song names
    temp_df = temp_df.drop_duplicates(subset='track_name')

    # Select top N songs
    result = temp_df[['track_name', 'artists', 'track_genre',
                      'popularity', 'similarity_score']].head(top_n)

    # Reset index
    result = result.reset_index(drop=True)

    return result


# Example
normal_recommendation('Du Hast', 10)


,track_name,artists,track_genre,popularity,similarity_score
0,Be My Lover,La Bouche,techno,0.68,0.999113
1,Radio,Rammstein,industrial,0.70,0.998908
2,No Money,Galantis,house,0.68,0.998778
3,Hail to the King,Avenged Sevenfold,metal,0.76,0.998723
4,Links 2 3 4,Rammstein,industrial,0.65,0.998713
5,I'm Every Woman,Chaka Khan,disco,0.66,0.998665
6,Enter Sandman,Metallica,hard-rock,0.81,0.998610
7,Rain In Ibiza,Felix Jaehn;The Stickmen Project;Calum Scott,german,0.74,0.998543
8,Free Yourself,Jessie Ware,british,0.65,0.998522
9,LO$ER=LO♡ER,TOMORROW X TOGETHER,k-pop,0.75,0.998512


### 2.2 Top songs of an artist

In [ ]:
def artist_songs(artist_name, top_n):
    # Find songs by the artist
    artist_songs = df_rec[df_rec['artists'].str.lower().str.contains(artist_name.lower())]

    if artist_songs.empty:
        print(f'No songs found for artist "{artist_name}".')
        return None

    return artist_songs[['track_name', 'artists', 'track_genre', 'popularity']].drop_duplicates(subset="track_name").sort_values(by='popularity', ascending=False).head(top_n).reset_index(drop=True)

# Example
artist_songs('michael jackson', 10)

### 2.3 Top songs in a genre

In [ ]:
def recommend_genres(genre, top_n):
    # Filter songs of the specified genre
    genre_songs = df_rec[df_rec['track_genre'].str.lower() == genre.lower()]
    if genre_songs.empty:
        print(f'Genre "{genre}" not found.')
        return None
    
    top_songs = genre_songs.drop_duplicates(subset='track_name').sort_values(by='popularity', ascending=False).head(top_n)
    return top_songs[['track_name', 'artists', 'popularity']].reset_index(drop=True)

# Example
recommend_genres('pop', 20)


### 2.4 Mood-Based Playlist generator   

In [ ]:
def mood_based(mood_type, top_n):
    rules = {
        'workout': (df['energy'] > 0.7) & (df['tempo'] > 0.6),
        'study':   (df['instrumentalness'] > 0.4) & (df['energy'] < 0.5),
        'relax':   (df['acousticness'] > 0.6) & (df['energy'] < 0.4),
        'party':   (df['danceability'] > 0.7) & (df['energy'] > 0.6),
        'happy':   (df['valence'] > 0.7) & (df['energy'] > 0.5),
        'sad':     (df['valence'] < 0.3) & (df['energy'] < 0.5),
    }
    if mood_type not in rules:
        print(f'Mood type "{mood_type}" not recognized. Choose from: {list(rules.keys())}')
        return None
    filtered_songs = df[rules[mood_type]]
    top_songs = filtered_songs.sort_values(by='popularity', ascending=False).drop_duplicates(subset='track_name').head(top_n)

    return top_songs[['track_name', 'artists', 'popularity']].reset_index(drop=True)  
            

# Example   
mood_based('party', 10)

## Clustering the songs

In [ ]:
audio_features = ['danceability', 'energy', 'valence', 'acousticness','tempo', 'loudness', 'speechiness', 'instrumentalness']

# sample 20000 songs
df_cluster = df.sample(20000, random_state=42).reset_index(drop=True)
cluster_scaler = MinMaxScaler()
X_cluster = cluster_scaler.fit_transform(df_cluster[audio_features])
X_cluster.shape


### Finding Optimal K using Elbow method

In [ ]:

inertia = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_cluster)
    inertia.append(km.inertia_)
plt.figure(figsize=(8,5))
plt.plot(K_range, inertia, marker='o')
plt.title('Elbow Method for Optimal K') 
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.tight_layout()
plt.show()

### K-Means clustering

In [ ]:

k=5
kmeans = KMeans(n_clusters=k, random_state=42)
df_cluster['cluster'] = kmeans.fit_predict(X_cluster)
df_cluster.head()
cluster_names = {
    0: 'Party Songs',
    1: 'Workout Songs',
    2: 'Calm Acoustic',
    3: 'Emotional Songs',
    4: 'Relaxing Songs'
}
df_cluster['cluster_name'] = df_cluster['cluster'].map(cluster_names)
joblib.dump(kmeans, 'models/kmeans.pkl')

pd.DataFrame(df_cluster[["cluster", "cluster_name"]].value_counts())


###
 PCA Visualization

In [ ]:

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster)
plt.figure(figsize=(8,6))

figure = px.scatter(df_cluster, x=X_pca[:, 0], y=X_pca[:, 1], 
    color='cluster_name',
    title='PCA Visualization of Clusters',
    labels={'x': 'First Principal Component', 'y': 'Second Principal Component'},
    hover_data=['track_name', 'artists'] )

figure.show()
 

In [ ]:
# Explained Variance Ratio
explained_variance = pca.explained_variance_ratio_
print("Explained Variance Ratio by PCA components:", explained_variance)

# 3. CLUSTER BASED RECOMMENDATION SYSTEM
Get the recommendations of a song from within it's cluster.

In [ ]:
def cluster_recommendation(song_name, top_n):
    matches = df_cluster[df_cluster['track_name'].str.lower() == song_name.lower()]
    if matches.empty:
        print(f'Song "{song_name}" not found in cluster dataset.')
        return None

    cluster_label = matches['cluster'].values[0]

    # Recommend songs from the same cluster, excluding the input song
    cluster_songs = df_cluster[df_cluster['cluster'] == cluster_label]

    # Remove songs with the same name (case-insensitive)
    cluster_songs = cluster_songs[cluster_songs['track_name'].str.lower() != song_name.lower()]

    # Keep only songs with popularity >= 0.65
    cluster_songs = cluster_songs[cluster_songs['popularity'] >= 0.65]

    # Remove duplicate song names
    cluster_songs = cluster_songs.drop_duplicates(subset='track_name')

    # Sort by popularity and select top N
    top_songs = cluster_songs.drop_duplicates(subset='track_name').sort_values(by='popularity', ascending=False).head(top_n)

    return top_songs[['track_name', 'artists', 'track_genre', 'popularity']].reset_index(drop=True)

# Example
cluster_recommendation('thriller', 10)

#  HIT SONG PREDICTION 

In [ ]:
## Hit probability prediction for a new song
def predict_hit_probability(song_features):
    # Ensure the input features are in the same order as training
    feature_order = joblib.load('models/feature_columns.pkl')
    song_vector = np.array([song_features[feature] for feature in feature_order]).reshape(1, -1)

    # Scale the features
    scaler = joblib.load('models/scaler.pkl')
    song_vector_scaled = scaler.transform(song_vector)

    # Load the trained model
    model = joblib.load('models/hit_predictor.pkl')

    # Predict hit probability
    hit_probability = model.predict_proba(song_vector_scaled)[0][1]

    return hit_probability

# Example usage
new_song = {
    'danceability': 0.82,
    'energy': 0.88,
    'loudness': -5,
    'speechiness': 0.06,
    'acousticness': 0.12,
    'instrumentalness': 0.00,
    'liveness': 0.18,
    'valence': 0.79,
    'tempo': 126,
    'duration_mins': 3.1
}
probability = predict_hit_probability(new_song)
print(f"Predicted hit probability: {probability:.2%}")


## HYBRID RECOMMENDATION SYSTEM

In [ ]:

def hybrid_recommendation(song_name, top_n, min_popularity=0.65, w_popularity=0.5, w_similarity=0.3, w_cluster=0.2):
    # Get content-based recommendations
    cosine_recs = normal_recommendation(song_name, top_n*2)  # Get more to allow for filtering

    if cosine_recs is None:
        return None

    # Get cluster-based recommendations
    cluster_recs = cluster_recommendation(song_name, top_n*2)

    if cluster_recs is None:
        return None

    # Merge recommendations
    merged_recs = pd.merge(cosine_recs, cluster_recs, on=['track_name', 'artists', 'track_genre', 'popularity'], how='outer', suffixes=('_content', '_cluster'))

    # Fill NaN similarity scores with 0
    merged_recs['similarity_score'] = merged_recs['similarity_score'].fillna(0)

    # Calculate combined score
    merged_recs['combined_score'] = (
        w_popularity * merged_recs['popularity']
        + w_similarity * merged_recs['similarity_score']
        + w_cluster * (merged_recs['similarity_score'] > 0).astype(int)
    )

    # Filter by minimum popularity
    filtered_recs = merged_recs[merged_recs['popularity'] >= min_popularity]

    # Sort by combined score and take top_n
    final_recs = filtered_recs.drop_duplicates(subset='track_name').sort_values(by='combined_score', ascending=False).head(top_n)

    return final_recs[['track_name', 'artists', 'track_genre', 'popularity', 'combined_score']].reset_index(drop=True)

# Example
hybrid_recommendation('thriller', 10)
    

# Full Comparision of all 3

In [ ]:
def comparision(song_name, top_n):
    print(f"Comparing recommendations for: {song_name}\n")
    
    nr = normal_recommendation(song_name, top_n)
    cr = cluster_recommendation(song_name, top_n)
    hr = hybrid_recommendation(song_name, top_n)

    print("Content-Based Recommendations:")
    print(nr.to_string(index=False))
    print("\nCluster-Based Recommendations:")
    print(cr.to_string(index=False))
    print("\nHybrid Recommendations:")
    print(hr.to_string(index=False))
# Example
comparision('dream on', 5)